<a href="https://colab.research.google.com/github/Statofthe7/ZeroShotMultilingualSentimentAnalysisSOVvsSVO/blob/main/MultilingualSentimentAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !apt-get update # Up to date list of packages
# !apt-get install -y mecab libmecab-dev # Mecab and its libraries for Fugashi

In [ ]:
!pip install datasets pandas transformers torch sentencepiece
# !pip install fugashi[unidic-lite]

In [ ]:
# from fugashi import Tagger
from transformers import AutoTokenizer
from transformers import pipeline
import warnings
import pandas as pd
from datasets import load_dataset, concatenate_datasets
from tqdm import tqdm # for progress bar
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from scipy import stats
import spacy
import numpy as np


In [ ]:
# This script loads the Allociné (French) and Japanese Generic Sentiment datasets
# It also calculates average and maximum "length" of text reviews in both datasets
# For French, length is count of words in each review
# For Japanese, lenght is count of morphemes or sub-word tokens

# mDeBERTa-v3 is a multilingual version of DeBERTa (Decoding-enhanced BERT with distangled attention) and is a medium sized pre-trained model
# As part of fine-tuning on mnli and xnli datasets, it has learned to reason about a premise's entailment
# and contradiction to a hypothesis (NLI-Natural Language Inference) across different languages
tokenizer = AutoTokenizer.from_pretrained("MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

# Load the French Dataset (Allociné Movie Reviews)
print("Loading French data...")
fr_data = load_dataset("allocine", split='test')

# Load the Japanese Sentiment Dataset
print("Loading Japanese data...")
jp_data = load_dataset("mteb/JapaneseSentimentClassification", split='test')

# Setup Japanese Tagger to breakup text into space separated (owakati) morphemes
#tagger = Tagger('-Owakati')

# Get token length in Japanese
def get_japanese_token_length(text):

    # Using fugashi to parse text to morphemes separated by spaces and split it to form a list
    # tokens = tagger.parse(text).split()

    # Using the transformer model's tokenizer (SentencePiece) to parse japanese text to sub-word tokens (similar to morphemes)
    tokens = tokenizer.encode(text)

    return len(tokens)

def get_stats(dataset, text_column, lang):
    df = pd.DataFrame(dataset)

    if lang == 'fr':
        # French: Split by spaces
        df['length'] = df[text_column].apply(lambda x: len(x.split()))
    else:
        # Japanese: Split by morphemes or sub-word tokens
        df['length'] = df[text_column].apply(get_japanese_token_length)

    return df['length'].mean(), df['length'].max()

fr_mean, fr_max = get_stats(fr_data, 'review', 'fr')
jp_mean, jp_max = get_stats(jp_data, 'text', 'jp')

print(f"\n--- Statistical Results ---")
print(f"French  - Avg Words per Review: {fr_mean:.2f} (Max: {fr_max})")
print(f"Japanese - Avg Morphs per Review: {jp_mean:.2f} (Max: {jp_max})")

In [ ]:
# This script initializes the classifier to perform zero shot classification

# Zero-Shot Pipeline is a NLP technique to perform classification on categories the pre-trained language model hasn't been trained on

classifier = pipeline("zero-shot-classification",
                      model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
                      device=-1) # Change to 0 if you have a GPU/CUDA

# Run Tests
def test_sentiment(text, lang_name):
    candidate_labels = ["positive", "negative"]

    # We use a 'hypothesis template' to guide the model's classification logic
    # This works across languages because the model is multilingual
    hypothesis_template = "The sentiment of this text is {}.";

    result = classifier(text, candidate_labels, hypothesis_template=hypothesis_template)

    print(f"\n--- {lang_name} Analysis ---")
    print(f"Text: {text}")
    print(f"Top Label: {result['labels'][0]} ({result['scores'][0]:.2%})")

fr_text = "Le film était absolument magnifique, bien que la fin soit un peu triste."
jp_text = "このカメラの画質は素晴らしいですが、バッテリーの持ちが悪いです。" # "Good quality, but bad battery"

test_sentiment(fr_text, "French")
test_sentiment(jp_text, "Japanese")

In [ ]:
# This script runs sentiment analysis on both the datasets and writes results into two csv files

# Ensures we have a fair 50/50 split of Positive and Negative samples
def get_balanced_data(dataset, num_per_class=50):
    neg = dataset.filter(lambda x: x['label'] == 0).shuffle(seed=42).select(range(num_per_class))
    pos = dataset.filter(lambda x: x['label'] == 1).shuffle(seed=42).select(range(num_per_class))

    print(f"  Retrieved {len(pos)} positive samples and {len(neg)} negative samples (target: {num_per_class} per class).")

    return concatenate_datasets([neg, pos]).shuffle(seed=42)


balanced_fr_data = get_balanced_data(fr_data)
balanced_jp_data = get_balanced_data(jp_data)

def run_experiment(dataset, text_col, label_col, lang_name, num_samples=100):
    results = []
    print(f"Analyzing {lang_name}...")

    df_sample = pd.DataFrame(dataset).sample(num_samples, random_state=42)

    for _, row in tqdm(df_sample.iterrows(), total=num_samples):
        raw_text = row[text_col]

        # Calculate Length based on language
        if lang_name == "Japanese":
           # processed_text = tagger.parse(raw_text)
            processed_text = raw_text
            token_length = len(tokenizer.encode(raw_text))
        else:
            processed_text = raw_text
            token_length = len(raw_text.split())

        true_label_idx = row[label_col]
        true_label = "positive" if true_label_idx == 1 else "negative"

        # Run AI prediction
        output = classifier(processed_text, ["positive", "negative"],
                                    hypothesis_template="The sentiment is {}.")

        pred_label = output['labels'][0]
        confidence = output['scores'][0]

        results.append({
            "raw_text": raw_text,
            "processed_text": processed_text,
            "token_length": token_length,
            "true_label": true_label,
            "pred_label": pred_label,
            "correct": true_label == pred_label,
            "confidence": confidence
        })

    return pd.DataFrame(results)

# Execute
df_fr = run_experiment(balanced_fr_data, 'review', 'label', "French")
df_jp = run_experiment(balanced_jp_data, 'text', 'label', "Japanese")

# Save to CSV
df_fr.to_csv("french_results.csv", index=False)
df_jp.to_csv("japanese_results.csv", index=False)

print("\nAccuracy French:", df_fr['correct'].mean())
print("Accuracy Japanese:", df_jp['correct'].mean())

In [ ]:
#This script charts confusion matrices for French and Japanese sentiment analysis

def plot_research_results(csv_file, lang_name):
    df = pd.read_csv(csv_file)

    # Create the Confusion Matrix
    # This shows: True Positives, False Positives, True Negatives, False Negatives
    cm = confusion_matrix(df['true_label'], df['pred_label'], labels=["positive", "negative"])

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=["Positive", "Negative"],
                yticklabels=["Positive", "Negative"])

    plt.title(f'Confusion Matrix: {lang_name} Sentiment Analysis')
    plt.ylabel('Actual Label')
    plt.xlabel('Predicted Label')
    if lang_name == "Japanese":
      plt.savefig('figure1_confusion_japanese.jpg', dpi=300, bbox_inches='tight')
    else:
      plt.savefig('figure2_confusion_french.jpg', dpi=300, bbox_inches='tight')
    plt.show()

# Run for both
plot_research_results("french_results.csv", "French")
plot_research_results("japanese_results.csv", "Japanese")

In [ ]:
# This script shows a comparison chart for accuracy of the results
acc_fr = pd.read_csv("french_results.csv")['correct'].mean()
acc_jp = pd.read_csv("japanese_results.csv")['correct'].mean()

plt.bar(['French', 'Japanese'], [acc_fr, acc_jp], color=['#3498db', '#e74c3c'])
plt.ylabel('Accuracy Score')
plt.title('Zero-Shot Sentiment Accuracy: French vs. Japanese')
plt.ylim(0, 1) # Set scale from 0 to 100%
plt.show()



In [ ]:
# This script is to visually see if token length affects model accuracy differently in
# French (SVO) vs. Japanese (SOV).

def plot_length_vs_accuracy(csv_file, lang_name):
    df = pd.read_csv(csv_file)

    # Create length "bins" (e.g., 0-10 words, 10-20, etc.)
    # This makes the trend much easier to see than a scatter plot
    df['length_bin'] = pd.cut(df['token_length'],bins=[0, 20, 40, 60, 100, 200, float('inf')],
                              labels=['(0-20)', '(20-40)', '(40-60)', '(60-100)', '(100-200)', '(200+)'])

    plt.figure(figsize=(10, 6))

    # Plotting mean accuracy for each bin
    sns.barplot(data=df, x='length_bin', y='correct', hue='length_bin', palette='magma', errorbar=None, legend=False)

    plt.title(f'Accuracy by Model Token Count: {lang_name}')
    plt.ylabel('Accuracy Rate')
    plt.xlabel('Number of Sub-word Tokens')
    plt.ylim(0, 1.1)

    # Add a horizontal line for the overall average
    plt.axhline(df['correct'].mean(), color='red', linestyle='--', label='Avg Accuracy')
    plt.legend()
    if lang_name == "Japanese":
      plt.savefig('figure4_length_accuracy_japanese.jpg', dpi=300, bbox_inches='tight')
    else:
      plt.savefig('figure3_length_accuracy_french.jpg', dpi=300, bbox_inches='tight')
    plt.show()

# Run for both results files
plot_length_vs_accuracy("french_results.csv", "French")
plot_length_vs_accuracy("japanese_results.csv", "Japanese")

In [ ]:
# This script shows statistical summary table with mean accuracy, correlation r
# and significance p for French and Japanese

def generate_statistical_summary(french_csv, japanese_csv):
    # Load your results
    df_fr = pd.read_csv(french_csv)
    df_jp = pd.read_csv(japanese_csv)

    summary_data = []

    for df, lang in [(df_fr, "French"), (df_jp, "Japanese")]:
        # Mean Accuracy
        mean_acc = df['correct'].mean()

        # Point-Biserial Correlation (r and p)
        # We correlate 'token_length' (continuous) with 'correct' (binary)
        # r (Correlation Coefficient) ranges from -1 to +1.
        # r - A value close to +1 indicates strong positive correlation. This means as length of text review increases, the likelihood of predicting sentiment also incresases
        # r - A value close to -1 indicates negative correlation. This means as length of text review increases, the likelihood of predicting sentiment decreases
        # r - A value close to 0 indicates little to no linear correlation
        # p (P-Value) shows if the observed correlation between token length and accuracy of predicted sentiment is statistically significant or likely due to random chance
        # p - If less than the conventional significance level of 0.05, the correlation is considered statistically significant. This means there is a strong correlation that likely exists in the larger population
        # p - If greater than 0.05, the correlation is generally not considered statistically significant. This means the weak correlation might be random variation

        r_val, p_val = stats.pointbiserialr(df['correct'], df['token_length'])

        summary_data.append({
            "Language": lang,
            "Mean Accuracy": f"{mean_acc:.2%}",
            "Correlation (r)": f"{r_val:.3f}",
            "P-Value": f"{p_val:.4f}",
            "Significance": "Significant" if p_val < 0.05 else "Not Significant"
        })

    # Create the final table
    summary_table = pd.DataFrame(summary_data)
    return summary_table

# Execute
results_table = generate_statistical_summary("french_results.csv", "japanese_results.csv")
print(results_table)

# Save for review
results_table.to_csv("statistical_summary.csv", index=False)